# Notebook 08 — Évaluation comparative (6 méthodes)

**Prérequis** : `test.json` + les JSON dans `results/` :
- `baseline_predictions.json`, `rag_predictions.json`, `finetuned_predictions.json`
- `raft_predictions.json`, `rerank_predictions.json`, `function_calling_predictions.json`

**Sorties** : `final_report.json`, figures dans `results/plots/`.


## 0. Montage Google Drive

In [ ]:
# Montage du Drive et définition du chemin de base du projet
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 1. Dépendances (métriques & plots)

In [ ]:
# Installation des bibliothèques d'évaluation et de visualisation
!pip install -q faiss-cpu sentence-transformers bert-score rouge-score matplotlib transformers accelerate peft bitsandbytes nltk

## 2. Imports et chemins

In [ ]:
# Imports évaluation (sans Unsloth / LoRA / FAISS)
import os, json, time, string, re
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter, defaultdict
from bert_score import score as bert_score_fn
from rouge_score import rouge_scorer as _rouge_scorer
from tqdm.notebook import tqdm
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize

for _pkg in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
    try:
        nltk.download(_pkg, quiet=True)
    except Exception:
        pass

PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
MODELS_PATH    = os.path.join(BASE_PATH, 'models', 'lora_adapter')
FAISS_PATH     = os.path.join(BASE_PATH, 'models', 'faiss_index')
RESULTS_PATH   = os.path.join(BASE_PATH, 'results')
PLOTS_PATH     = os.path.join(BASE_PATH, 'results', 'plots')

for path in [RESULTS_PATH, PLOTS_PATH]:
    os.makedirs(path, exist_ok=True)

EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
TOP_K       = 5
MAX_SEQ_LEN = 2048

_ROUGE = _rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

METHODS_ORDER    = ["Baseline", "RAG", "Fine-tuné", "RAFT", "Rerank", "Function-calling"]
STRATA_ORDER     = ["récent", "intermédiaire", "fondamental"]
QTYPES_ORDER     = ["factuel", "synthese", "comprehension"]
METHOD_COLORS    = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974', '#8c564b']
STRATA_COLORS    = ['#e74c3c', '#f39c12', '#2ecc71']
QTYPE_COLORS     = ['#3498db', '#9b59b6', '#1abc9c']

BS_THRESHOLDS = [0.80, 0.85, 0.90]
BS_THRESHOLD  = 0.85
BOOTSTRAP_N   = 1000
BOOTSTRAP_SEED = 42

print("Configuration évaluation chargée (notebook 08).")
print(f"  Seuils accuracy : {[int(t*100) for t in BS_THRESHOLDS]}%  (principal : {int(BS_THRESHOLD*100)}%)")
print(f"  Bootstrap IC 95% : {BOOTSTRAP_N} réplicatas (seed {BOOTSTRAP_SEED}) sur scores par paire")
print(f"  Résultats    : {RESULTS_PATH}")


## 3. Chargement des fichiers de prédictions


In [ ]:
# Chargement des fichiers JSON depuis Drive
def load_json(path, label=""):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  [OK] {label or path} ({len(data)} entrées)")
        return data
    except FileNotFoundError:
        print(f"  [MANQUANT] {path}")
        return []
    except json.JSONDecodeError as e:
        print(f"  [ERROR JSON] {path} : {e}")
        return []

print("Chargement des fichiers...")
test_data             = load_json(os.path.join(PROCESSED_PATH,  'test.json'),                   'test.json')
baseline_predictions  = load_json(os.path.join(RESULTS_PATH,    'baseline_predictions.json'),   'baseline_predictions.json')
rag_predictions       = load_json(os.path.join(RESULTS_PATH,    'rag_predictions.json'),        'rag_predictions.json')
finetuned_predictions = load_json(os.path.join(RESULTS_PATH,    'finetuned_predictions.json'),  'finetuned_predictions.json')
raft_predictions      = load_json(os.path.join(RESULTS_PATH,    'raft_predictions.json'),        'raft_predictions.json')
rerank_predictions    = load_json(os.path.join(RESULTS_PATH,    'rerank_predictions.json'),      'rerank_predictions.json')
fc_predictions        = load_json(os.path.join(RESULTS_PATH,    'function_calling_predictions.json'), 'function_calling_predictions.json')

print(f"\nTest set : {len(test_data)} questions")


## 4. Évaluation des 6 méthodes


In [ ]:
# Métriques : EM, F1, ROUGE-L, BERTScore, METEOR, hallucination, latence, confiance
# + intervalles de confiance bootstrap (95 %) sur scores par paire (BERTScore, METEOR, F1, ROUGE-L).
def normalize_text(text):
    text = (text or "").lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation))
    stop = {'a','an','the','le','la','les','un','une','des'}
    return ' '.join(t for t in text.split() if t not in stop)

def exact_match(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def f1_token(pred, gold):
    pred_tok = normalize_text(pred).split()
    gold_tok = normalize_text(gold).split()
    if not pred_tok or not gold_tok:
        return 0.0
    common = Counter(pred_tok) & Counter(gold_tok)
    n = sum(common.values())
    if n == 0:
        return 0.0
    p = n / len(pred_tok)
    r = n / len(gold_tok)
    return 2 * p * r / (p + r)

def rouge_l(pred, gold):
    if not (pred or "").strip() or not (gold or "").strip():
        return 0.0
    return _ROUGE.score(gold, pred)['rougeL'].fmeasure

def meteor_one(pred, gold):
    """METEOR (NLTK) : signature `meteor_score(references, hypothesis)` avec references=liste de listes de tokens."""
    text_p = (pred or "").strip()
    text_g = (gold or "").strip()
    if not text_p or not text_g:
        return 0.0

    def _tokens(txt):
        low = txt.lower()
        try:
            return word_tokenize(low, language="french")
        except (LookupError, TypeError, ValueError):
            try:
                return word_tokenize(low)
            except Exception:
                return re.findall(r"\w+", low, flags=re.UNICODE)

    ref = _tokens(text_g)
    hyp = _tokens(text_p)
    if not ref or not hyp:
        return 0.0
    try:
        return float(meteor_score([ref], hyp, gamma=0.0))
    except Exception:
        return 0.0

def compute_bert_score(preds, refs, batch_size=32):
    """BERTScore : textes non vides (strip) et longueur bornée pour éviter candidats vides côté lib."""
    def _clip(t):
        s = (t or "").strip()
        if not s:
            s = " "
        return s[:4000]
    preds = [_clip(p) for p in preds]
    refs = [_clip(r) for r in refs]
    try:
        _, _, F = bert_score_fn(preds, refs, lang="fr",
                                model_type="distilbert-base-multilingual-cased",
                                batch_size=batch_size, verbose=False)
        return F.tolist()
    except Exception as e:
        print(f"  [ERROR] BERTScore : {e}")
        return [0.0] * len(preds)

def hallucination_score(pred, context):
    if not (pred or "").strip() or not (context or "").strip():
        return 1.0
    return 1.0 - rouge_l(pred, context)

def bootstrap_mean_ci(values, n_rep=None, seed=None, alpha=0.05):
    """IC bootstrap sur la moyenne (percentiles 2.5 / 97.5). `values` : scores par exemple (même échelle que la moyenne affichée)."""
    n_rep = int(n_rep or BOOTSTRAP_N)
    seed = BOOTSTRAP_SEED if seed is None else seed
    arr = np.asarray(values, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return None, None
    rng = np.random.default_rng(seed)
    means = np.empty(n_rep, dtype=np.float64)
    n = arr.size
    for i in range(n_rep):
        idx = rng.integers(0, n, size=n)
        means[i] = float(arr[idx].mean())
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)

_EMPTY_METRICS = {
    "exact_match": 0.0, "f1": 0.0, "bertscore": 0.0, "rouge_l": 0.0, "meteor": 0.0,
    "hallucination": 0.0, "latency_ms": 0.0, "confidence_mean": None,
    "bertscore_ci95_low": None, "bertscore_ci95_high": None,
    "meteor_ci95_low": None, "meteor_ci95_high": None,
    "f1_ci95_low": None, "f1_ci95_high": None,
    "rouge_l_ci95_low": None, "rouge_l_ci95_high": None,
    "bootstrap_n": BOOTSTRAP_N,
    **{f"acc_bs{int(t*100)}": 0.0 for t in BS_THRESHOLDS},
    "bs_by_recency": {}, "bs_by_qtype": {},
    "hall_by_recency": {}, "hall_by_qtype": {},
    "bs_by_dstype": {}, "hall_by_dstype": {},
    "bs_multisauts_simple": None, "bs_multisauts_complexe": None,
}

def canon_recency(raw):
    """Aligne test.json (souvent ASCII) sur les libellés FR utilisés dans les figures."""
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return "inconnu"
    k = str(raw).strip().lower()
    if not k:
        return "inconnu"
    alias = {
        "recent": "récent", "récent": "récent",
        "intermediaire": "intermédiaire", "intermédiaire": "intermédiaire",
        "fondamental": "fondamental", "fundamental": "fondamental",
    }
    return alias.get(k, k)

def evaluate_method(predictions, method_name, test_lookup=None):
    if not predictions:
        print(f"  [WARN] Aucune prédiction pour '{method_name}'")
        return {"method": method_name, "n": 0, **_EMPTY_METRICS}

    em_list, f1_list, rl_list, hall_list, lat_list = [], [], [], [], []
    meteor_list = []
    conf_list = []
    preds_list, refs_list = [], []
    by_recency = defaultdict(lambda: {"preds": [], "refs": [], "hall": []})
    by_qtype   = defaultdict(lambda: {"preds": [], "refs": [], "hall": []})
    by_dstype  = defaultdict(lambda: {"preds": [], "refs": [], "hall": []})
    hal_ms_s_preds, hal_ms_s_refs = [], []
    hal_ms_c_preds, hal_ms_c_refs = [], []

    for p in predictions:
        pred = (p.get('predicted_answer') or '').strip()
        gold = (p.get('true_answer') or '').strip()
        pair_id = p.get('pair_id', '')
        recency, qtype, context = 'inconnu', 'inconnu', ''
        item = None
        if test_lookup and pair_id in test_lookup:
            item = test_lookup[pair_id]
            recency = canon_recency(item.get('recency_category', 'inconnu'))
            qtype = item.get('question_type', 'inconnu')
            context = item.get('context', '')

        em_list.append(exact_match(pred, gold))
        f1_list.append(f1_token(pred, gold))
        rl_list.append(rouge_l(pred, gold))
        meteor_list.append(meteor_one(pred, gold))
        hall_list.append(hallucination_score(pred, context if context else gold))
        preds_list.append(pred if pred else " ")
        refs_list.append(gold if gold else " ")

        by_recency[recency]["preds"].append(preds_list[-1]); by_recency[recency]["refs"].append(refs_list[-1]); by_recency[recency]["hall"].append(hall_list[-1])
        by_qtype[qtype]["preds"].append(preds_list[-1]); by_qtype[qtype]["refs"].append(refs_list[-1]); by_qtype[qtype]["hall"].append(hall_list[-1])
        dstype = (item.get("dataset_type") if item else None) or p.get("dataset_type") or "inconnu"
        by_dstype[dstype]["preds"].append(preds_list[-1]); by_dstype[dstype]["refs"].append(refs_list[-1]); by_dstype[dstype]["hall"].append(hall_list[-1])
        if item and item.get("dataset_type") == "multisauts":
            qt0 = (item.get("question_type") or "").lower()
            if qt0 == "simple":
                hal_ms_s_preds.append(preds_list[-1]); hal_ms_s_refs.append(refs_list[-1])
            elif qt0 in ("complexe", "complex"):
                hal_ms_c_preds.append(preds_list[-1]); hal_ms_c_refs.append(refs_list[-1])

        if p.get('latency_ms', 0) > 0:
            lat_list.append(p['latency_ms'])
        if p.get("confidence") is not None:
            conf_list.append(float(p.get("confidence")))

    print(f"  Calcul BERTScore pour '{method_name}' ({len(preds_list)} paires)...")
    bs_list = compute_bert_score(preds_list, refs_list)
    bs_by_recency = {cat: round(np.mean(compute_bert_score(d['preds'], d['refs'])) * 100, 2) if d["preds"] else 0.0 for cat, d in by_recency.items()}
    bs_by_qtype = {qt: round(np.mean(compute_bert_score(d['preds'], d['refs'])) * 100, 2) if d["preds"] else 0.0 for qt, d in by_qtype.items()}
    hall_by_recency = {k: round(np.mean(v["hall"]) * 100, 2) for k, v in by_recency.items()}
    hall_by_qtype = {k: round(np.mean(v["hall"]) * 100, 2) for k, v in by_qtype.items()}
    accuracy_by_threshold = {f"acc_bs{int(t*100)}": round(sum(s >= t for s in bs_list) / len(bs_list) * 100, 2) for t in BS_THRESHOLDS}

    f1_pct = np.array(f1_list, dtype=np.float64) * 100.0
    rl_pct = np.array(rl_list, dtype=np.float64) * 100.0
    bs_pct = np.array(bs_list, dtype=np.float64) * 100.0
    met_pct = np.array(meteor_list, dtype=np.float64) * 100.0

    print(f"  Bootstrap IC 95% (N={BOOTSTRAP_N}) pour '{method_name}'...")
    b_lo, b_hi = bootstrap_mean_ci(bs_pct)
    m_lo, m_hi = bootstrap_mean_ci(met_pct)
    f_lo, f_hi = bootstrap_mean_ci(f1_pct)
    r_lo, r_hi = bootstrap_mean_ci(rl_pct)

    bs_ms_simple = round(float(np.mean(compute_bert_score(hal_ms_s_preds, hal_ms_s_refs))) * 100, 2) if hal_ms_s_preds else None
    bs_ms_complex = round(float(np.mean(compute_bert_score(hal_ms_c_preds, hal_ms_c_refs))) * 100, 2) if hal_ms_c_preds else None

    return {
        "method": method_name,
        "exact_match": round(np.mean(em_list) * 100, 2),
        "f1": round(float(np.mean(f1_pct)), 2),
        "bertscore": round(float(np.mean(bs_pct)), 2),
        "rouge_l": round(float(np.mean(rl_pct)), 2),
        "meteor": round(float(np.mean(met_pct)), 2),
        "hallucination": round(np.mean(hall_list) * 100, 2),
        "latency_ms": round(np.mean(lat_list), 1) if lat_list else 0,
        "confidence_mean": round(np.mean(conf_list), 4) if conf_list else None,
        "n": len(predictions),
        "bootstrap_n": BOOTSTRAP_N,
        "bertscore_ci95_low": round(b_lo, 2) if b_lo is not None else None,
        "bertscore_ci95_high": round(b_hi, 2) if b_hi is not None else None,
        "meteor_ci95_low": round(m_lo, 2) if m_lo is not None else None,
        "meteor_ci95_high": round(m_hi, 2) if m_hi is not None else None,
        "f1_ci95_low": round(f_lo, 2) if f_lo is not None else None,
        "f1_ci95_high": round(f_hi, 2) if f_hi is not None else None,
        "rouge_l_ci95_low": round(r_lo, 2) if r_lo is not None else None,
        "rouge_l_ci95_high": round(r_hi, 2) if r_hi is not None else None,
        **accuracy_by_threshold,
        "bs_by_recency": bs_by_recency,
        "bs_by_qtype": bs_by_qtype,
        "hall_by_recency": hall_by_recency,
        "hall_by_qtype": hall_by_qtype,
        "bs_by_dstype": {k: round(np.mean(compute_bert_score(v["preds"], v["refs"])) * 100, 2) if v["preds"] else 0.0 for k, v in by_dstype.items()},
        "hall_by_dstype": {k: round(np.mean(v["hall"]) * 100, 2) for k, v in by_dstype.items()},
        "bs_multisauts_simple": bs_ms_simple,
        "bs_multisauts_complexe": bs_ms_complex,
    }

print("Fonctions d'évaluation prêtes : METEOR + IC bootstrap 95 % (BERTScore, METEOR, F1, ROUGE-L).")

In [ ]:
# Calcul des métriques pour les 6 méthodes
test_lookup = {item.get('pair_id', ''): item for item in test_data}

print("Évaluation en cours...\n")
methods_to_eval = [
    ("Baseline", baseline_predictions),
    ("RAG", rag_predictions),
    ("Fine-tuné", finetuned_predictions),
    ("RAFT", raft_predictions),
    ("Rerank", rerank_predictions),
    ("Function-calling", fc_predictions),
]
all_results = []
for method_name, preds in methods_to_eval:
    result = evaluate_method(preds, method_name, test_lookup=test_lookup)
    all_results.append(result)
    acc85 = result.get('acc_bs85', 0)
    conf = result.get("confidence_mean")
    conf_txt = f"{conf:.3f}" if conf is not None else "N/A"
    b_lo, b_hi = result.get("bertscore_ci95_low"), result.get("bertscore_ci95_high")
    bs_ci = f"[{b_lo:.1f},{b_hi:.1f}]" if b_lo is not None and b_hi is not None else "N/A"
    print(f"  {method_name:<18} EM={result['exact_match']:5.1f}%  F1={result['f1']:5.1f}%  "
          f"BS={result['bertscore']:5.1f}% {bs_ci}  METEOR={result.get('meteor',0):5.1f}%  "
          f"RL={result['rouge_l']:5.1f}%  Acc@85={acc85:5.1f}%  Hall={result['hallucination']:5.1f}%  "
          f"Lat={result['latency_ms']:.0f}ms  Conf={conf_txt}")

print("\nÉvaluation terminée.")
METHODS_ORDER = [r["method"] for r in all_results]
METHOD_COLORS = ['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974','#8c564b'][:len(METHODS_ORDER)]


In [ ]:
# Tableau comparatif pandas
def _ci_str(lo, hi):
    if lo is None or hi is None:
        return "N/A"
    return f"[{lo:.1f}, {hi:.1f}]"

df = pd.DataFrame([{
    "Méthode": r['method'],
    "Exact Match (%)": r['exact_match'],
    "F1 (%)": r['f1'],
    "F1 IC95%": _ci_str(r.get('f1_ci95_low'), r.get('f1_ci95_high')),
    "BERTScore (%)": r['bertscore'],
    "BERTScore IC95%": _ci_str(r.get('bertscore_ci95_low'), r.get('bertscore_ci95_high')),
    "ROUGE-L (%)": r['rouge_l'],
    "ROUGE-L IC95%": _ci_str(r.get('rouge_l_ci95_low'), r.get('rouge_l_ci95_high')),
    "METEOR (%)": r.get('meteor', 0),
    "METEOR IC95%": _ci_str(r.get('meteor_ci95_low'), r.get('meteor_ci95_high')),
    "Acc@85% (%)": r.get('acc_bs85', 0),
    "Hallucination (%)": r['hallucination'],
    "Latence moy. (ms)": r['latency_ms'],
    "Confiance moy.": r.get('confidence_mean', None),
    "N": r['n'],
} for r in all_results]).set_index("Méthode")

print("=" * 90)
print("TABLEAU COMPARATIF — Méthodes disponibles")
print("=" * 90)
display(
    df.style
      .highlight_max(subset=["Exact Match (%)","F1 (%)","BERTScore (%)","ROUGE-L (%)","METEOR (%)","Acc@85% (%)"], color='lightgreen')
      .highlight_min(subset=["Hallucination (%)","Latence moy. (ms)"], color='lightblue')
      .format(
          subset=["Exact Match (%)","F1 (%)","BERTScore (%)","ROUGE-L (%)","METEOR (%)","Acc@85% (%)","Hallucination (%)","Latence moy. (ms)","Confiance moy.","N"],
          precision=3,
      )
)
print("\nConfiance moy. : moyenne du proxy de confiance token-level (si présent dans les prédictions).")

In [ ]:
# Sauvegarde du rapport final en JSON
report = {
    "generated_at": time.strftime('%Y-%m-%dT%H:%M:%S'),
    "n_test_samples": len(test_data),
    "bs_thresholds_used": [int(t * 100) for t in BS_THRESHOLDS],
    "bootstrap_n": BOOTSTRAP_N,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "bootstrap_ci_level": 0.95,
    "methods": all_results
}
report_path = os.path.join(RESULTS_PATH, 'final_report.json')

try:
    with open(report_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(f"Rapport sauvegardé : {report_path}  ({os.path.getsize(report_path)/1024:.1f} Ko)")
except Exception as e:
    print(f"[ERROR] Sauvegarde rapport : {e}")

## 5. Visualisations


In [ ]:
# ── Figure 1 : Métriques globales (EM / F1 / BERTScore / ROUGE-L) ──────────────
methods  = [r['method']    for r in all_results]
metrics  = {
    "Exact Match (%)": [r['exact_match'] for r in all_results],
    "F1 (%)":            [r['f1']          for r in all_results],
    "BERTScore (%)":   [r['bertscore']   for r in all_results],
    "ROUGE-L (%)":     [r['rouge_l']     for r in all_results],
    "METEOR (%)":      [r.get('meteor', 0) for r in all_results],
}
colors5  = ['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974']

x     = np.arange(len(methods))
width = 0.15
fig1, ax1 = plt.subplots(figsize=(13, 6))
for i, (label, vals) in enumerate(metrics.items()):
    offset = (i - 2) * width
    bars = ax1.bar(x + offset, vals, width, label=label, color=colors5[i], alpha=0.87)
    for bar in bars:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.1f}',
                 ha='center', va='bottom', fontsize=7.5)

ax1.set_xlabel('Méthode', fontsize=12)
ax1.set_ylabel('Score (%)', fontsize=12)
ax1.set_title('Figure 1 — Métriques globales par méthode\n(Exact Match, F1, BERTScore, ROUGE-L, METEOR)',
              fontsize=13, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(methods, fontsize=11)
ax1.set_ylim(0, 115); ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)
plt.tight_layout()
p1 = os.path.join(PLOTS_PATH, 'fig1_global_metrics.png')
fig1.savefig(p1, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 1 sauvegardée : {p1}")

In [ ]:
# ── Figure 2 : BERTScore par strate temporelle × méthode ───────────────────────
fig2, ax2 = plt.subplots(figsize=(11, 6))
x2    = np.arange(len(METHODS_ORDER))
w2    = 0.22
for i, (cat, col) in enumerate(zip(STRATA_ORDER, STRATA_COLORS)):
    vals = [r.get('bs_by_recency', {}).get(cat, 0) for r in all_results]
    offset = (i - 1) * w2
    bars = ax2.bar(x2 + offset, vals, w2, label=cat.capitalize(), color=col, alpha=0.87)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax2.text(bar.get_x() + bar.get_width()/2, h + 0.4, f'{h:.1f}',
                     ha='center', va='bottom', fontsize=7.5)

ax2.set_xlabel('Méthode', fontsize=12)
ax2.set_ylabel('BERTScore (%)', fontsize=12)
ax2.set_title('Figure 2 — BERTScore par strate temporelle × méthode\n'
              '(récent 2022-2025 / intermédiaire 2019-2022 / fondamental pré-2019)',
              fontsize=12, fontweight='bold')
ax2.set_xticks(x2); ax2.set_xticklabels(METHODS_ORDER, fontsize=11)
ax2.set_ylim(0, 115); ax2.legend(title='Strate', fontsize=9)
ax2.grid(axis='y', alpha=0.3)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
plt.tight_layout()
p2 = os.path.join(PLOTS_PATH, 'fig2_bertscore_by_recency.png')
fig2.savefig(p2, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 2 sauvegardée : {p2}")

In [ ]:
# ── Figure 3 : Performance par type de question × méthode ──────────────────────
fig3, ax3 = plt.subplots(figsize=(11, 6))
x3 = np.arange(len(METHODS_ORDER))
w3 = 0.22
for i, (qt, col) in enumerate(zip(QTYPES_ORDER, QTYPE_COLORS)):
    vals = [r.get('bs_by_qtype', {}).get(qt, 0) for r in all_results]
    offset = (i - 1) * w3
    bars = ax3.bar(x3 + offset, vals, w3, label=qt.capitalize(), color=col, alpha=0.87)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax3.text(bar.get_x() + bar.get_width()/2, h + 0.4, f'{h:.1f}',
                     ha='center', va='bottom', fontsize=7.5)

ax3.set_xlabel('Méthode', fontsize=12)
ax3.set_ylabel('BERTScore (%)', fontsize=12)
ax3.set_title('Figure 3 — BERTScore par type de question × méthode\n'
              '(factuel / synthèse / compréhension)',
              fontsize=12, fontweight='bold')
ax3.set_xticks(x3); ax3.set_xticklabels(METHODS_ORDER, fontsize=11)
ax3.set_ylim(0, 115); ax3.legend(title='Type de question', fontsize=9)
ax3.grid(axis='y', alpha=0.3)
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
plt.tight_layout()
p3 = os.path.join(PLOTS_PATH, 'fig3_bertscore_by_qtype.png')
fig3.savefig(p3, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 3 sauvegardée : {p3}")

In [ ]:
# ── Figure 4 : Latence moyenne par méthode ─────────────────────────────────────
lat_vals = [r['latency_ms'] for r in all_results]
fig4, ax4 = plt.subplots(figsize=(8, 5))
bars4 = ax4.bar(METHODS_ORDER, lat_vals, color=METHOD_COLORS, alpha=0.87, width=0.5)
for bar, val in zip(bars4, lat_vals):
    ax4.text(bar.get_x() + bar.get_width()/2, val + max(lat_vals)*0.01,
             f'{val:,.0f} ms', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax4.set_xlabel('Méthode', fontsize=12)
ax4.set_ylabel('Latence moyenne (ms)', fontsize=12)
ax4.set_title('Figure 4 — Latence moyenne par méthode', fontsize=13, fontweight='bold')
ax4.set_ylim(0, max(lat_vals) * 1.25 if lat_vals else 1)
ax4.grid(axis='y', alpha=0.3)
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)
plt.tight_layout()
p4 = os.path.join(PLOTS_PATH, 'fig4_latency.png')
fig4.savefig(p4, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 4 sauvegardée : {p4}")

In [ ]:
# ── Figure 5 : Taux d'hallucination estimé par méthode ─────────────────────────
hall_vals = [r['hallucination'] for r in all_results]
fig5, axes = plt.subplots(1, 2, figsize=(14, 5))

# 5a — Hallucination globale
bars5 = axes[0].bar(METHODS_ORDER, hall_vals, color=METHOD_COLORS, alpha=0.87, width=0.5)
for bar, val in zip(bars5, hall_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Méthode', fontsize=12)
axes[0].set_ylabel('Taux d\'hallucination estimé (%)', fontsize=11)
axes[0].set_title('5a — Global', fontsize=11, fontweight='bold')
axes[0].set_ylim(0, max(hall_vals) * 1.35 if hall_vals else 100)
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# 5b — Hallucination par strate temporelle
x5b = np.arange(len(STRATA_ORDER))
w5b = 0.18
for i, (method, col) in enumerate(zip(METHODS_ORDER, METHOD_COLORS)):
    r = next((r for r in all_results if r['method'] == method), {})
    vals5b = [r.get('hall_by_recency', {}).get(cat, 0) for cat in STRATA_ORDER]
    axes[1].bar(x5b + (i - 1.5) * w5b, vals5b, w5b, label=method, color=col, alpha=0.87)
axes[1].set_xlabel('Strate temporelle', fontsize=12)
axes[1].set_ylabel('Taux d\'hallucination (%)', fontsize=11)
axes[1].set_title('5b — Par strate temporelle', fontsize=11, fontweight='bold')
axes[1].set_xticks(x5b)
axes[1].set_xticklabels([s.capitalize() for s in STRATA_ORDER], fontsize=10)
_hall5b_max = max((v for r in all_results for v in (r.get('hall_by_recency') or {}).values()), default=0)
axes[1].set_ylim(0, max(110, _hall5b_max * 1.15, 1.0))
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

fig5.suptitle('Figure 5 — Taux d\'hallucination estimé\n(proxy : 1 − ROUGE-L(prédit, contexte source))',
              fontsize=12, fontweight='bold')
plt.tight_layout()
p5 = os.path.join(PLOTS_PATH, 'fig5_hallucination.png')
fig5.savefig(p5, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 5 sauvegardée : {p5}")

In [ ]:
# ── Figure 6 : Trade-off Qualité vs Latence (scatter) ──────────────────────────
fig6, ax6 = plt.subplots(figsize=(8, 6))
for r, col in zip(all_results, METHOD_COLORS):
    ax6.scatter(r['latency_ms'], r['bertscore'],
                s=220, color=col, zorder=5, edgecolors='white', linewidths=1.5)
    ax6.annotate(r['method'],
                 xy=(r['latency_ms'], r['bertscore']),
                 xytext=(8, 4), textcoords='offset points',
                 fontsize=11, fontweight='bold', color=col)

ax6.set_xlabel('Latence moyenne (ms)  ← plus rapide', fontsize=12)
ax6.set_ylabel('BERTScore (%)  ↑ meilleure qualité', fontsize=12)
ax6.set_title('Figure 6 — Trade-off Qualité vs Latence\n(idéal : coin haut-gauche)',
              fontsize=12, fontweight='bold')
ax6.grid(alpha=0.3)
ax6.spines['top'].set_visible(False); ax6.spines['right'].set_visible(False)

# Zone idéale
ymin, ymax = ax6.get_ylim()
xmin, xmax = ax6.get_xlim()
ax6.annotate('← Idéal (rapide + précis)', xy=(xmin, ymax),
             xytext=(xmin + (xmax-xmin)*0.02, ymax - (ymax-ymin)*0.05),
             fontsize=9, color='green', style='italic')

plt.tight_layout()
p6 = os.path.join(PLOTS_PATH, 'fig6_quality_vs_latency.png')
fig6.savefig(p6, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 6 sauvegardée : {p6}")

## 5b. Tableau croisé méthodes × dataset_type


In [ ]:
# ── Tableau croisé : BERTScore par méthode × dataset_type ────────────────────
_DATASET_ORDER = ["technique", "multisauts", "temporel", "juridique"]
_type_counts = Counter(it.get("dataset_type") for it in test_data if it.get("dataset_type"))
DATASET_TYPES = [dt for dt in _DATASET_ORDER if _type_counts.get(dt, 0) > 0]
if not DATASET_TYPES:
    DATASET_TYPES = ["technique", "multisauts", "temporel"]
print(f"dataset_type présents dans test.json : {dict(_type_counts)} → colonnes heatmap : {DATASET_TYPES}")

crosstable_rows = []
for r in all_results:
    row = {"Méthode": r["method"]}
    for dt in DATASET_TYPES:
        row[dt.capitalize()] = r.get("bs_by_dstype", {}).get(dt, 0.0)
    row["Moyenne"] = r["bertscore"]
    crosstable_rows.append(row)

df_cross = pd.DataFrame(crosstable_rows).set_index("Méthode")

print("\nBERTScore (%) par méthode × dataset_type")
print("=" * 60)
print(df_cross.to_string())

# Highlight best per column
def highlight_max_col(col):
    return ['font-weight: bold; color: green' if v == col.max() else '' for v in col]

print("\nHAL multisauts — BERTScore simple vs complexe (voir aussi Figure 8) :")
for r in all_results:
    bs = r.get("bs_by_qtype", {})
    s  = bs.get("simple", 0)
    c  = bs.get("complexe", 0)
    print(f"  {r['method']:<12}  simple={s:.1f}%  complexe={c:.1f}%  delta={c-s:+.1f}%")

In [ ]:
# ── Figure 9 : Accuracy par seuil BERTScore ──────────────────────────────────
threshold_labels = [f"≥{int(t*100)}%" for t in BS_THRESHOLDS]
threshold_keys   = [f"acc_bs{int(t*100)}" for t in BS_THRESHOLDS]

fig9, axes9 = plt.subplots(1, 2, figsize=(14, 5))

# 9a — Barres groupées : accuracy à 3 seuils pour les 4 méthodes
x9    = np.arange(len(METHODS_ORDER))
w9    = 0.22
shade_colors = ['#2ecc71', '#f39c12', '#e74c3c']  # vert, orange, rouge selon seuil
for i, (key, label, col) in enumerate(zip(threshold_keys, threshold_labels, shade_colors)):
    vals = [r.get(key, 0) for r in all_results]
    offset = (i - 1) * w9
    bars = axes9[0].bar(x9 + offset, vals, w9, label=label, color=col, alpha=0.80)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            axes9[0].text(bar.get_x() + bar.get_width()/2, h + 0.8,
                          f'{h:.0f}', ha='center', va='bottom', fontsize=8)

axes9[0].set_xlabel('Méthode', fontsize=12)
axes9[0].set_ylabel('Accuracy (%)', fontsize=12)
axes9[0].set_title('9a — % réponses "correctes" selon le seuil BERTScore', fontsize=11, fontweight='bold')
axes9[0].set_xticks(x9); axes9[0].set_xticklabels(METHODS_ORDER, fontsize=11)
axes9[0].set_ylim(0, 115); axes9[0].legend(title='Seuil BERTScore', fontsize=9)
axes9[0].grid(axis='y', alpha=0.3)
axes9[0].spines['top'].set_visible(False); axes9[0].spines['right'].set_visible(False)

# 9b — Courbe d'accuracy vs seuil (de 0.70 à 0.95) pour chaque méthode
# On ne peut calculer ça que si on a accès aux scores individuels — ici on affiche les 3 seuils calculés
thresholds_pct = [int(t * 100) for t in BS_THRESHOLDS]
for r, col in zip(all_results, METHOD_COLORS):
    acc_vals = [r.get(f"acc_bs{t}", 0) for t in thresholds_pct]
    axes9[1].plot(thresholds_pct, acc_vals, marker='o', color=col,
                  linewidth=2, markersize=8, label=r['method'])
    for x_pt, y_pt in zip(thresholds_pct, acc_vals):
        axes9[1].annotate(f'{y_pt:.0f}%', xy=(x_pt, y_pt),
                          xytext=(3, 5), textcoords='offset points', fontsize=8, color=col)

axes9[1].set_xlabel('Seuil BERTScore (%)', fontsize=12)
axes9[1].set_ylabel('% réponses au-dessus du seuil', fontsize=12)
axes9[1].set_title('9b — Courbe accuracy vs seuil par méthode', fontsize=11, fontweight='bold')
axes9[1].set_xticks(thresholds_pct)
axes9[1].set_xticklabels([f'{t}%' for t in thresholds_pct])
axes9[1].set_ylim(0, 115); axes9[1].legend(fontsize=9)
axes9[1].grid(alpha=0.3)
axes9[1].spines['top'].set_visible(False); axes9[1].spines['right'].set_visible(False)

fig9.suptitle('Figure 9 — Pourcentage de réponses "correctes" par seuil BERTScore\n'
              '(Acc@X% = % réponses avec BERTScore ≥ X% — plus souple qu\'Exact Match)',
              fontsize=12, fontweight='bold')
plt.tight_layout()
p9 = os.path.join(PLOTS_PATH, 'fig9_accuracy_by_threshold.png')
fig9.savefig(p9, dpi=150, bbox_inches='tight'); plt.show()
print(f"Figure 9 sauvegardée : {p9}")

# Afficher le résumé textuel
print(f"\n--- Résumé Accuracy@{int(BS_THRESHOLD*100)}% ---")
for r in all_results:
    acc = r.get(f'acc_bs{int(BS_THRESHOLD*100)}', 0)
    print(f"  {r['method']:<12} : {acc:.1f}% des réponses ont BERTScore ≥ {int(BS_THRESHOLD*100)}%")

In [ ]:
# ── Figure 7 : Heatmap BERTScore méthodes × dataset_type ─────────────────────
import numpy as np

heatmap_data = df_cross[[dt.capitalize() for dt in DATASET_TYPES]].values.astype(float)

fig7, ax7 = plt.subplots(figsize=(8, 5))
im = ax7.imshow(heatmap_data, cmap="YlOrRd", aspect="auto",
                vmin=max(0, heatmap_data.min() - 5),
                vmax=min(100, heatmap_data.max() + 5))

ax7.set_xticks(range(len(DATASET_TYPES)))
ax7.set_xticklabels([dt.capitalize() for dt in DATASET_TYPES], fontsize=12)
ax7.set_yticks(range(len(METHODS_ORDER)))
ax7.set_yticklabels(METHODS_ORDER, fontsize=12)

for i in range(len(METHODS_ORDER)):
    for j in range(len(DATASET_TYPES)):
        val = heatmap_data[i, j]
        text_color = "white" if val < (heatmap_data.min() + heatmap_data.max()) / 2 else "black"
        ax7.text(j, i, f"{val:.1f}%", ha="center", va="center",
                 fontsize=11, fontweight="bold", color=text_color)

plt.colorbar(im, ax=ax7, label="BERTScore (%)")
ax7.set_title("Figure 7 — BERTScore par méthode × dataset_type\n"
              "(colonnes = types présents dans test.json ; multisauts = HAL FR)",
              fontsize=12, fontweight="bold")
fig7.tight_layout()
p7 = os.path.join(PLOTS_PATH, "fig7_heatmap_methodes_datasets.png")
fig7.savefig(p7, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure 7 sauvegardée : {p7}")

# ── Figure 8 : Arxiv simple vs complexe (bar chart) ──────────────────────────
fig8, ax8 = plt.subplots(figsize=(8, 5))
x8 = np.arange(len(METHODS_ORDER))
w8 = 0.35
simple_vals  = [r.get("bs_multisauts_simple")    or 0 for r in all_results]
complex_vals = [r.get("bs_multisauts_complexe") or 0 for r in all_results]

bars_s = ax8.bar(x8 - w8/2, simple_vals,  w8, label="Simple (1 fait)",           color="#4C72B0", alpha=0.85)
bars_c = ax8.bar(x8 + w8/2, complex_vals, w8, label="Complexe (multi-sauts)", color="#DD8452", alpha=0.85)
for bars in [bars_s, bars_c]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax8.text(bar.get_x() + bar.get_width()/2, h + 0.4, f"{h:.1f}",
                     ha="center", va="bottom", fontsize=8)

ax8.set_xticks(x8); ax8.set_xticklabels(METHODS_ORDER, fontsize=11)
ax8.set_ylabel("BERTScore (%)", fontsize=12)
ax8.set_title("Figure 8 — HAL multisauts : questions simples vs complexes\n"
              "(0 pour complexe si aucun exemple `complexe` au test — regénérer 02 si besoin)",
              fontsize=12, fontweight="bold")
ax8.legend(fontsize=10); ax8.set_ylim(0, 115)
ax8.grid(axis="y", alpha=0.3)
ax8.spines["top"].set_visible(False); ax8.spines["right"].set_visible(False)
fig8.tight_layout()
p8 = os.path.join(PLOTS_PATH, "fig8_simple_vs_complexe.png")
fig8.savefig(p8, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure 8 sauvegardée : {p8}")

## 6. Résumé final


In [ ]:
# Récapitulatif des artefacts d'évaluation (notebook 08)
print("=" * 70)
print("RÉSUMÉ FINAL — Pipeline complet 'Intégration de nouvelles informations dans les LLMs'")
print("=" * 70)

all_output_files = [
    ("01_scraping",        os.path.join(BASE_PATH, 'data', 'raw', 'wikipedia.json')),
    ("01_scraping",        os.path.join(BASE_PATH, 'data', 'raw', 'arxiv.json')),
    ("02_dataset_builder", os.path.join(BASE_PATH, 'data', 'processed', 'train.json')),
    ("02_dataset_builder", os.path.join(BASE_PATH, 'data', 'processed', 'test.json')),
    ("03_baseline_rag",    os.path.join(BASE_PATH, 'results', 'baseline_predictions.json')),
    ("03_baseline_rag",    os.path.join(BASE_PATH, 'results', 'rag_predictions.json')),
    ("03_baseline_rag",    os.path.join(FAISS_PATH, 'index.faiss')),
    ("04_finetuning",      os.path.join(BASE_PATH, 'results', 'finetuned_predictions.json')),
    ("07_function_calling",     os.path.join(BASE_PATH, 'results', 'function_calling_predictions.json')),
    ("08_evaluation",    report_path),
] + [("08_evaluation", p) for p in [p1, p2, p3, p4, p5, p6, p7, p8, p9]]

print(f"\n{'Notebook':<22} {'Taille':>8}   Fichier")
print("-" * 70)
for nb, fpath in all_output_files:
    try:
        size = f"{os.path.getsize(fpath)/1024:>6.1f} Ko"
    except Exception:
        size = "  N/A   "
    print(f"{nb:<22} {size}   {fpath}")

# Meilleures performances par métrique
print("\n--- Meilleures performances ---")
best_em   = max(all_results, key=lambda x: x['exact_match'])
best_f1   = max(all_results, key=lambda x: x['f1'])
best_bs   = max(all_results, key=lambda x: x['bertscore'])
best_rl   = max(all_results, key=lambda x: x['rouge_l'])
best_met  = max(all_results, key=lambda x: x.get('meteor', 0))
best_hall = min(all_results, key=lambda x: x['hallucination'])
best_lt   = min([r for r in all_results if r['latency_ms'] > 0], key=lambda x: x['latency_ms'], default=all_results[0])

print(f"  Exact Match      : {best_em['method']:<12} {best_em['exact_match']:.1f}%")
print(f"  F1 Token         : {best_f1['method']:<12} {best_f1['f1']:.1f}%")
print(f"  BERTScore        : {best_bs['method']:<12} {best_bs['bertscore']:.1f}%")
print(f"  ROUGE-L          : {best_rl['method']:<12} {best_rl['rouge_l']:.1f}%")
print(f"  METEOR           : {best_met['method']:<12} {best_met.get('meteor',0):.1f}%")
print(f"  Hallucination \u2193  : {best_hall['method']:<12} {best_hall['hallucination']:.1f}%")
print(f"  Latence min      : {best_lt['method']:<12} {best_lt['latency_ms']:.0f} ms")

print("\n✔ Pipeline complet terminé. Tous les résultats sont sur Google Drive.")
print(f"  → {BASE_PATH}")
print("=" * 70)